# Rule Generation and Risk Analysis System

This notebook implements a system that:
1. Loads question-response data from a spreadsheet
2. Generates rules based on responses using LLM
3. Analyzes risk data against generated rules
4. Provides recommendations for improvement

In [1]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog
import openai
import anthropic
import json
from pathlib import Path
from dotenv import load_dotenv
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import os

# Load environment variables from .env file in the same folder as the notebook
dotenv_path = Path(__file__).parent / '.env' if '__file__' in globals() else Path().cwd() / '.env'
load_dotenv(dotenv_path)

# Set up AI provider API keys from environment
openai.api_key = os.getenv('OPENAI_API_KEY')
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

In [2]:
# Function to select input file
def select_file(title="Select a file"):
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    file_path = filedialog.askopenfilename(
        title=title,
        filetypes=[("Excel files", "*.xlsx *.xls")]
    )
    return file_path

# Function to load question-response data
def load_question_responses(file_path):
    if not file_path:
        return None
    df = pd.read_excel(file_path)
    return df

# Function to generate rules using Anthropic LLM
def generate_rules(question, responses, provider="anthropic", output_format="json"):
    if provider != "anthropic":
        raise ValueError("This notebook is set to use Anthropic only.")

    if not os.getenv('ANTHROPIC_API_KEY'):
        raise ValueError("Anthropic API key not found in .env file. Please add ANTHROPIC_API_KEY to your .env file.")

    if output_format == "json":
        prompt = f"""
        You are an expert risk analyst. Your response must start and end with a valid JSON array of rule objects. Do not include any text, explanation, or markdown before or after the array.

        Question: {question}

        Responses:
        {responses}

        Instructions:
        - Carefully analyze the responses and extract the implicit human reasoning and decision criteria for risk assessment.
        - For each unique pattern or interpretation, create a rule object that is actionable and can be used to evaluate new risk data.
        - Respond ONLY with a valid JSON array of rule objects. Do not include any explanation, markdown, or text outside the JSON array.
        - Each rule object must have:
          - rule_id: a unique identifier
          - description: a clear, human-centric description of the rule
          - validation_criteria: specific, actionable criteria to check in risk data
        """
    elif output_format == "text":
        prompt = f"""
        You are an expert risk analyst. Read the following question and its set of human responses, and infer the underlying human logic and criteria used to judge risks. For each distinct pattern, interpretation, or principle you find, write a clear, human-readable rule in plain text. Do not use JSON or any structured format.

        Question: {question}

        Responses:
        {responses}

        Instructions:
        - Carefully analyze the responses and extract the implicit human reasoning and decision criteria for risk assessment.
        - For each unique pattern or interpretation, write a rule in plain English, starting each rule on a new line.
        - Do not use JSON, markdown, or any structured format. Only plain text rules.
        """
    else:
        raise ValueError("Unsupported output_format. Use 'json' or 'text'.")

    try:
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        raw_text = response.content[0].text
        print(f"Raw Anthropic response for question '{question}':\n{raw_text}\n")
        if output_format == "json":
            import re
            match = re.search(r'(\[\s*{.*?}\s*\])', raw_text, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(1))
                except Exception as json_error:
                    print(f"JSON decode error: {json_error}")
                    print(f"Extracted JSON string: {match.group(1)}")
                    return None
            else:
                print("No JSON array found in response. Full raw response:")
                print(raw_text)
                return None
        elif output_format == "text":
            # Return plain text rules, split by lines
            rules = [line.strip() for line in raw_text.split('\n') if line.strip()]
            return rules
    except Exception as e:
        print(f"Error generating rules with Anthropic: {e}")
        return None

In [3]:
# Main execution flow
provider = "anthropic"  # LLM provider is set to Anthropic
rules_file = "250905 Risk Heuristics survey download.xlsx"  # File in same folder as notebook

if rules_file:
    # Load question-response data
    question_data = load_question_responses(rules_file)
    
    if question_data is not None:
        print("\nGenerating rules from responses...")
        all_rules = []
        all_text_rules = []
        
        # Generate rules for each question
        for column in question_data.columns:
            responses = question_data[column].dropna().tolist()
            # Generate JSON rules
            rules = generate_rules(column, responses, provider=provider, output_format="json")
            if rules:
                all_rules.extend(rules)
            # Generate plain text rules
            text_rules = generate_rules(column, responses, provider=provider, output_format="text")
            if text_rules:
                all_text_rules.append({"question": column, "rules": text_rules})
        
        print(f"\nGenerated {len(all_rules)} JSON rules.")
        print(f"Generated plain text rules for {len(all_text_rules)} questions.")
        
        # Save JSON rules to file
        rules_output = Path(rules_file).parent / "generated_rules.json"
        with open(rules_output, 'w') as f:
            json.dump(all_rules, f, indent=2)
        print(f"\nJSON rules saved to: {rules_output}")
        
        # Save plain text rules to file
        text_rules_output = Path(rules_file).parent / "generated_rules.txt"
        with open(text_rules_output, 'w') as f:
            for entry in all_text_rules:
                f.write(f"Question: {entry['question']}\n")
                for rule in entry['rules']:
                    f.write(f"- {rule}\n")
                f.write("\n")
        print(f"Plain text rules saved to: {text_rules_output}")
else:
    print("No file selected. Process cancelled.")


Generating rules from responses...
Raw Anthropic response for question 'You know it’s not really a risk when…':
```json
[
  {
    "rule_id": "R001",
    "description": "A risk is not valid if it has already occurred or is currently impacting the project",
    "validation_criteria": "Check if the risk description uses past tense verbs, indicates current impact, or shows the event has already materialized. Keywords: 'has happened', 'is impacting', 'already occurred', 'currently active'"
  },
  {
    "rule_id": "R002",
    "description": "A risk is not valid if it lacks uncertainty (probability is 100% or near-certain)",
    "validation_criteria": "Check if probability is above 85-90% or described as 'certain', 'definite', 'will happen'. If certainty exists, it should be treated as an issue or planning item"
  },
  {
    "rule_id": "R003",
    "description": "A risk is not valid if it cannot be articulated in the standard risk format (Cause-Event-Impact)",
    "validation_criteria": "Ver

In [4]:
# Load rules from JSON file
rules_path = 'generated_rules.json'  # Update with your rules file path
with open(rules_path, 'r') as f:
    rules = json.load(f)

# Convert to DataFrame for easier processing
rules_df = pd.DataFrame(rules)
rules_df.head()

,rule_id,description,validation_criteria
0,R001,Schedule risk is the most frequently selected ...,"When evaluating project risks, prioritize sche..."
1,R002,Resource Availability is the second most commo...,Assess whether the risk involves staffing leve...
2,R003,Technical Uncertainty ranks as a significant r...,Identify risks involving unproven technologies...
3,R004,"Supplier Delay appears moderately, indicating ...",Evaluate risks stemming from vendor performanc...
4,R005,"Cost Overrun is selected less frequently, sugg...",Classify as Cost Overrun only when the primary...


# Usage Instructions

1. Before running this notebook, make sure to:
   - Install all required packages
   - Set up your OpenAI API key in the `generate_rules` function
   - Prepare your input spreadsheets

2. Input Spreadsheet Requirements:
   - Rules spreadsheet: Each column should be a question, and rows contain responses
   - Risk data spreadsheet: Two sheets
     - Sheet 1: Risk definitions (with columns for Risk ID, Risk Description)
     - Sheet 2: Mitigations

3. Running the Notebook:
   - Execute all cells in order
   - Follow the file selection prompts
   - Wait for the analysis to complete

4. Outputs:
   - generated_rules.json: Contains the LLM-generated rules
   - risk_analysis_results.xlsx: Contains the analysis results and recommendations

In [7]:
# Apply generated rules to risks and mitigations, classify as good/ok/poor, and save results
import numpy as np
from IPython.display import display

# Use the correct risk file name
risk_file = "PDAC Hack Risk data v5.1.xlsx"

# Load generated rules
rules_path = Path(rules_file).parent / "generated_rules.json"
with open(rules_path, 'r') as f:
    generated_rules = json.load(f)

# Load risk and mitigation sheets
risk_data = pd.read_excel(risk_file, sheet_name=None)
risks_df = risk_data.get('Risks')
mitigations_df = risk_data.get('Mitigations')

# Helper function to check if a risk/mitigation meets a rule
# This is a simple keyword-based check; for more advanced logic, use NLP or custom functions
def meets_rule(entry, rule):
    criteria = rule.get('validation_criteria', '').lower()
    description = str(entry).lower()
    # Check if any keyword from criteria is in the entry description
    keywords = [kw.strip() for kw in criteria.replace(',', ' ').split() if len(kw) > 2]
    return any(kw in description for kw in keywords)

# Classify entries

def classify_entries(df, rules):
    classifications = []
    for idx, row in df.iterrows():
        entry_text = ' '.join([str(val) for val in row.values if pd.notnull(val)])
        matches = sum(meets_rule(entry_text, rule) for rule in rules)
        if matches >= max(2, int(0.5 * len(rules))):
            classification = 'good'
        elif matches > 0:
            classification = 'ok'
        else:
            classification = 'poor'
        classifications.append(classification)
    return classifications

if risks_df is not None:
    risks_df['Classification'] = classify_entries(risks_df, generated_rules)
    print("\nRisk Classification Results:")
    display(risks_df)
if mitigations_df is not None:
    mitigations_df['Classification'] = classify_entries(mitigations_df, generated_rules)
    print("\nMitigation Classification Results:")
    display(mitigations_df)

# Save results to new Excel file
output_path = Path(risk_file).parent / "risk_analysis_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    if risks_df is not None:
        risks_df.to_excel(writer, sheet_name='Risk Classification', index=False)
    if mitigations_df is not None:
        mitigations_df.to_excel(writer, sheet_name='Mitigation Classification', index=False)
print(f"\nRisk and mitigation classification results saved to: {output_path}")


Risk Classification Results:


,Report Date,Organisation,Business Area,Portfolio ID,Project ID,Risk ID,Risk Unique ID,Risk Type,Cont,Status,...,PreMit_Sched_Impact_Min,PreMit_Sched_Impact_Most_Likely,PreMit_Sched_Impact_Max,PreMit_Sched_Impact,PreMit_Sched_Impact_Prob.1,PostMit_Sched_Impact_Min,PostMit_Sched_Impact_Most_Likely,PostMit_Sched_Impact_Max,PostMit_Sched_Impact,Classification
0,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_001,RISK015,Project_001_RISK015,Threat,Project Contingency,Open,...,111.0,124.0,149.0,126.000000,0.1,13.0,45.0,71.0,44.000000,good
1,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_001,RISK014,Project_001_RISK014,Opportunitity,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good
2,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_001,RISK016,Project_001_RISK016,Threat,Project Contingency,Open,...,88.0,138.0,147.0,131.166667,0.3,12.0,52.0,74.0,49.000000,good
3,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_001,RISK013,Project_001_RISK013,Threat,Project Contingency,Open,...,104.0,134.0,148.0,131.333333,0.3,20.0,48.0,77.0,48.166667,good
4,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_001,RISK008,Project_001_RISK008,Threat,Project Contingency,Open,...,103.0,135.0,148.0,131.833333,0.4,29.0,58.0,69.0,55.000000,good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17344,2018-08-22 00:00:00,ORG_001,BA_002,Portfolio_007,Project_029,RISK016,Project_029_RISK016,Threat,Project Contingency,Open,...,80.0,130.0,146.0,124.333333,0.8,80.0,130.0,146.0,124.333333,good
17345,2018-08-22 00:00:00,ORG_001,BA_002,Portfolio_007,Project_029,RISK011,Project_029_RISK011,Threat,Project Contingency,Open,...,94.0,124.0,154.0,124.000000,0.2,25.0,52.0,76.0,51.500000,good
17346,2018-08-22 00:00:00,ORG_001,BA_002,Portfolio_007,Project_029,RISK018,Project_029_RISK018,Threat,Project Contingency,Open,...,108.0,133.0,155.0,132.500000,0.9,108.0,133.0,155.0,132.500000,good
17347,2018-08-22 00:00:00,ORG_001,BA_002,Portfolio_007,Project_029,RISK020,Project_029_RISK020,Threat,Project Contingency,Open,...,100.0,125.0,142.0,123.666667,0.4,17.0,51.0,73.0,49.000000,good



Mitigation Classification Results:


,Serial,Report Date,Project ID,Risk ID,Unique_Mitigation_ID,Mit Action Owner,Action Description,Classification
0,1,2016-05-23 00:00:00,Project_001,RISK015,Project_001_RISK015_1,Person_674,Produce a lower level NRE plan.,good
1,2,2016-05-23 00:00:00,Project_001,RISK015,Project_001_RISK015_2,Person_766,Look at other Accreditation activity as a port...,good
2,3,2016-05-23 00:00:00,Project_001,RISK016,Project_001_RISK016_3,Person_674,Every technical event has a new booking code t...,good
3,4,2016-05-23 00:00:00,Project_001,RISK013,Project_001_RISK013_4,Person_674,Agree a list of named resources to be used for...,good
4,5,2016-05-23 00:00:00,Project_001,RISK008,Project_001_RISK008_5,Person_674,Negotiation with the customer.,good
...,...,...,...,...,...,...,...,...
15616,15617,2018-08-22 00:00:00,Project_029,RISK026,Project_029_RISK026_15617,Person_332,Provide Hw support throughout the procurement ...,good
15617,15618,2018-08-22 00:00:00,Project_029,RISK026,Project_029_RISK026_15618,Person_403,Structure the PO on a quality related payment ...,good
15618,15619,2018-08-22 00:00:00,Project_029,RISK027,Project_029_RISK027_15619,Person_332,Provide Hw support throughout the procurement ...,good
15619,15620,2018-08-22 00:00:00,Project_029,RISK027,Project_029_RISK027_15620,Person_403,Structure the PO on a quality related payment ...,good



Risk and mitigation classification results saved to: risk_analysis_results.xlsx


In [8]:
# Test/train/val split for risk data
from sklearn.model_selection import train_test_split

# Load risk data (assuming 'risks_df' already exists, otherwise load it)
# If not loaded, uncomment and adjust the following:
# risk_file = "PDAC Hack Risk data v5.1.xlsx"
# risk_data = pd.read_excel(risk_file, sheet_name=None)
# risks_df = risk_data.get('Risks')

# Split into train, validation, and test sets (60/20/20 split)
train_df, temp_df = train_test_split(risks_df, test_size=0.4, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Optionally display the splits
from IPython.display import display
print("\nTrain set:")
display(train_df)
print("\nValidation set:")
display(val_df)
print("\nTest set:")
display(test_df)

# Save splits to Excel for reference
split_output = Path(risk_file).parent / "risk_data_splits.xlsx"
with pd.ExcelWriter(split_output) as writer:
    train_df.to_excel(writer, sheet_name='Train', index=False)
    val_df.to_excel(writer, sheet_name='Validation', index=False)
    test_df.to_excel(writer, sheet_name='Test', index=False)
print(f"\nRisk data splits saved to: {split_output}")

Train set size: 10409
Validation set size: 3470
Test set size: 3470

Train set:


,Report Date,Organisation,Business Area,Portfolio ID,Project ID,Risk ID,Risk Unique ID,Risk Type,Cont,Status,...,PreMit_Sched_Impact_Min,PreMit_Sched_Impact_Most_Likely,PreMit_Sched_Impact_Max,PreMit_Sched_Impact,PreMit_Sched_Impact_Prob.1,PostMit_Sched_Impact_Min,PostMit_Sched_Impact_Most_Likely,PostMit_Sched_Impact_Max,PostMit_Sched_Impact,Classification
15340,2018-07-22 00:00:00,ORG_004,BA_007,Portfolio_019,Project_077,T-026,Project_077_T-026,Threat,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok
460,2016-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_038,RISK220,Project_038_RISK220,Threat,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good
4505,2017-02-20 00:00:00,ORG_001,BA_001,Portfolio_001,Project_018,OPP001,Project_018_OPP001,Opportunitity,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good
9541,2017-12-21 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK041-Risk-,Project_065_RISK041-Risk-,Threat,Project Contingency,Open,...,92.0,133.0,156.0,130.000000,0.3,16.0,41.0,61.0,40.166667,good
37,2016-05-23 00:00:00,ORG_001,BA_002,Portfolio_004,Project_007,R-074,Project_007_R-074,Threat,Project Contingency,Open,...,91.0,139.0,160.0,134.500000,0.3,3.0,43.0,65.0,40.000000,good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,2018-02-20 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK020,Project_065_RISK020,Threat,Management Contingency,Open,...,105.0,133.0,156.0,132.166667,0.1,32.0,53.0,71.0,52.500000,good
11964,2018-03-23 00:00:00,ORG_001,BA_003,Portfolio_013,Project_049,RISK084,Project_049_RISK084,Threat,Project Contingency,Open,...,114.0,133.0,158.0,134.000000,0.1,38.0,59.0,79.0,58.833333,good
5390,2017-03-23 00:00:00,ORG_001,BA_002,Portfolio_002,Project_028,Risk-003,Project_028_Risk-003,Threat,Project Contingency,Open,...,101.0,130.0,146.0,127.833333,0.9,101.0,130.0,146.0,127.833333,good
860,2016-06-23 00:00:00,ORG_001,BA_002,Portfolio_004,Project_023,Risk-021,Project_023_Risk-021,Opportunitity,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good



Validation set:


,Report Date,Organisation,Business Area,Portfolio ID,Project ID,Risk ID,Risk Unique ID,Risk Type,Cont,Status,...,PreMit_Sched_Impact_Min,PreMit_Sched_Impact_Most_Likely,PreMit_Sched_Impact_Max,PreMit_Sched_Impact,PreMit_Sched_Impact_Prob.1,PostMit_Sched_Impact_Min,PostMit_Sched_Impact_Most_Likely,PostMit_Sched_Impact_Max,PostMit_Sched_Impact,Classification
7309,2017-08-21 00:00:00,ORG_001,BA_003,Portfolio_013,Project_049,RISK136,Project_049_RISK136,Threat,Project Contingency,Open,...,88.0,133.0,160.0,130.000000,0.1,25.0,55.0,63.0,51.333333,good
9420,2017-12-21 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK099,Project_065_RISK099,Threat,Project Contingency,Open,...,80.0,132.0,150.0,126.333333,0.1,26.0,53.0,68.0,51.000000,good
9618,2017-12-21 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK342,Project_065_RISK342,Threat,Project Contingency,Open,...,84.0,124.0,144.0,120.666667,0.1,30.0,41.0,64.0,43.000000,good
16065,2018-07-22 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK511,Project_065_RISK511,Threat,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good
7202,2017-08-21 00:00:00,ORG_001,BA_002,Portfolio_002,Project_028,Risk-006,Project_028_Risk-006,Threat,Project Contingency,Open,...,116.0,125.0,156.0,128.666667,0.5,116.0,125.0,156.0,128.666667,good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1602,2016-07-21 00:00:00,ORG_001,BA_001,Portfolio_003,Project_011,Risk-103,Project_011_Risk-103,Threat,Project Contingency,Open,...,87.0,132.0,157.0,128.666667,0.9,87.0,132.0,157.0,128.666667,good
13864,2018-05-23 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK104,Project_065_RISK104,Threat,Project Contingency,Open,...,115.0,121.0,155.0,125.666667,0.2,30.0,46.0,71.0,47.500000,good
6607,2017-07-21 00:00:00,ORG_001,BA_001,Portfolio_003,Project_021,Risk-RO22,Project_021_Risk-RO22,Threat,Project Contingency,Open,...,120.0,131.0,155.0,133.166667,0.3,35.0,40.0,75.0,45.000000,good
14254,2018-05-23 00:00:00,ORG_001,BA_001,Portfolio_001,Project_038,RISK009,Project_038_RISK009,Threat,Project Contingency,Open,...,95.0,125.0,155.0,125.000000,0.8,95.0,125.0,155.0,125.000000,good



Test set:


,Report Date,Organisation,Business Area,Portfolio ID,Project ID,Risk ID,Risk Unique ID,Risk Type,Cont,Status,...,PreMit_Sched_Impact_Min,PreMit_Sched_Impact_Most_Likely,PreMit_Sched_Impact_Max,PreMit_Sched_Impact,PreMit_Sched_Impact_Prob.1,PostMit_Sched_Impact_Min,PostMit_Sched_Impact_Most_Likely,PostMit_Sched_Impact_Max,PostMit_Sched_Impact,Classification
16968,2018-08-22 00:00:00,ORG_001,BA_002,Portfolio_017,Project_062,Risk-021,Project_062_Risk-021,Threat,Project Contingency,Open,...,88.0,132.0,144.0,126.666667,0.3,26.0,59.0,66.0,54.666667,good
6827,2017-07-21 00:00:00,ORG_001,BA_001,Portfolio_001,Project_010,RISK80,Project_010_RISK80,Threat,Project Contingency,Open,...,80.0,126.0,151.0,122.500000,0.2,8.0,43.0,64.0,40.666667,good
3026,2016-10-21 00:00:00,ORG_001,BA_002,Portfolio_005,Project_042,Risk006,Project_042_Risk006,Threat,Project Contingency,Open,...,83.0,122.0,153.0,120.666667,0.8,83.0,122.0,153.0,120.666667,good
1486,2016-07-21 00:00:00,ORG_001,BA_002,Portfolio_002,Project_025,RS013,Project_025_RS013,Threat,Project Contingency,Open,...,83.0,136.0,159.0,131.000000,0.4,13.0,46.0,65.0,43.666667,ok
2114,2016-09-20 00:00:00,ORG_001,BA_001,Portfolio_001,Project_017,RISK045,Project_017_RISK045,Threat,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13848,2018-05-23 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK459,Project_065_RISK459,Threat,Project Contingency,Open,...,86.0,131.0,158.0,128.000000,0.2,36.0,44.0,60.0,45.333333,good
16453,2018-08-22 00:00:00,ORG_001,BA_003,Portfolio_006,Project_026,Risk-021,Project_026_Risk-021,Threat,Project Contingency,Open,...,120.0,134.0,141.0,132.833333,0.1,3.0,42.0,74.0,40.833333,good
16194,2018-07-22 00:00:00,ORG_002,BA_005,Portfolio_011,Project_065,RISK467,Project_065_RISK467,Threat,Project Contingency,Open,...,93.0,140.0,155.0,134.666667,0.1,8.0,52.0,68.0,47.333333,good
4026,2016-12-21 00:00:00,ORG_001,BA_001,Portfolio_003,Project_043,Risk-031,Project_043_Risk-031,Threat,Project Contingency,Open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,good



Risk data splits saved to: risk_data_splits.xlsx


In [9]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Logistic Regression Classification on Risk Data
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import json
from pathlib import Path

# Load risk data (if not already loaded)
risk_file = "risk_analysis_results.xlsx"
risk_data = pd.read_excel(risk_file, sheet_name=None)
risks_df = risk_data.get('Risks')

# Load generated rules
rules_file = "250905 Risk Heuristics survey download.xlsx"
rules_path = Path(rules_file).parent / "generated_rules.json"
with open(rules_path, 'r') as f:
    generated_rules = json.load(f)

# Helper function to check if a risk/mitigation meets a rule
def meets_rule(entry, rule):
    criteria = rule.get('validation_criteria', '').lower()
    description = str(entry).lower()
    keywords = [kw.strip() for kw in criteria.replace(',', ' ').split() if len(kw) > 2]
    return any(kw in description for kw in keywords)

def classify_entries(df, rules):
    classifications = []
    for idx, row in df.iterrows():
        entry_text = ' '.join([str(val) for val in row.values if pd.notnull(val)])
        matches = sum(meets_rule(entry_text, rule) for rule in rules)
        if matches >= max(2, int(0.5 * len(rules))):
            classification = 'good'
        elif matches > 0:
            classification = 'ok'
        else:
            classification = 'poor'
        classifications.append(classification)
    return classifications

# Run train/val/test split if not already done
try:
    train_df
    val_df
    test_df
except NameError:
    from sklearn.model_selection import train_test_split
    train_df, temp_df = train_test_split(risks_df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    print("Train/val/test split was missing and has now been created.")

# Ensure 'Classification' column exists in splits
for split_df in [train_df, val_df, test_df]:
    if 'Classification' not in split_df.columns:
        split_df['Classification'] = classify_entries(split_df, generated_rules)

# Prepare features and target
def prepare_features(df):
    X = df.select_dtypes(include=[np.number])
    # Fill NaN values in features with 0 (or another placeholder if preferred)
    X = X.fillna(0)
    return X

def prepare_target(df):
    y = df['Classification'].map({'good': 2, 'ok': 1, 'poor': 0})
    # Fill NaN values in target with -1 (or another placeholder if preferred)
    y = y.fillna(-1)
    return y

# Prepare train/val/test splits
X_train = prepare_features(train_df)
y_train = prepare_target(train_df)
X_val = prepare_features(val_df)
y_val = prepare_target(val_df)
X_test = prepare_features(test_df)
y_test = prepare_target(test_df)

# Train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate on validation and test sets
val_preds = model.predict(X_val)
test_preds = model.predict(X_test)

# Dynamically set labels and target_names for reports
unique_labels = sorted(set(y_train.unique()) | set(y_val.unique()) | set(y_test.unique()))
label_map = {0: 'poor', 1: 'ok', 2: 'good', -1: 'empty'}
target_names = [label_map.get(l, str(l)) for l in unique_labels]

print("\nValidation Classification Report:")
print(classification_report(y_val, val_preds, labels=unique_labels, target_names=target_names))
print("\nTest Classification Report:")
print(classification_report(y_test, test_preds, labels=unique_labels, target_names=target_names))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, test_preds, labels=unique_labels))


Validation Classification Report:
              precision    recall  f1-score   support

        poor       0.00      0.00      0.00         0
          ok       0.00      0.00      0.00        48
        good       0.99      1.00      0.99      3422

    accuracy                           0.99      3470
   macro avg       0.33      0.33      0.33      3470
weighted avg       0.97      0.99      0.98      3470


Test Classification Report:
              precision    recall  f1-score   support

        poor       0.00      0.00      0.00         0
          ok       0.00      0.00      0.00        57
        good       0.98      1.00      0.99      3413

    accuracy                           0.98      3470
   macro avg       0.33      0.33      0.33      3470
weighted avg       0.97      0.98      0.98      3470


Test Confusion Matrix:
[[   0    0    0]
 [   0    0   57]
 [   0    0 3413]]


C:\Users\sarah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sarah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sarah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classificati